# 🌧️ Uttar Pradesh Rainfall & Weather EDA (2005–2025)

**Dataset:** 565,210 daily weather records across 73 UP districts  
**Source:** NASA POWER Climate Data  
**Goal:** Understand rainfall patterns, temperature trends, and seasonal climate dynamics across UP

| Feature | Detail |
|---------|--------|
| Rows | 565,210 daily records |
| Columns | 20 features |
| Districts | 73 UP districts |
| Period | 2005–2025 (21 years) |
| Key Variable | PRECTOTCORR (Daily Rainfall in mm) |

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import os
os.makedirs('outputs', exist_ok=True)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('All libraries imported successfully!')


## 1. Load & Preview Data

In [ ]:
df = pd.read_csv('UP_rainfall_dataset.csv')

# Standardize district names
df['DISTRICT'] = df['DISTRICT'].str.strip().str.title()

# Create date column
df['DATE'] = pd.to_datetime(df[['YEAR','MO','DY']].rename(
    columns={'YEAR':'year','MO':'month','DY':'day'}))
df['SEASON'] = df['MO'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Pre-Monsoon', 4:'Pre-Monsoon', 5:'Pre-Monsoon',
    6:'Monsoon', 7:'Monsoon', 8:'Monsoon', 9:'Monsoon',
    10:'Post-Monsoon', 11:'Post-Monsoon'
})

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Districts: {df['DISTRICT'].nunique()} | Years: {df['YEAR'].min()}-{df['YEAR'].max()}")
df.head()


In [ ]:
print("Column Descriptions:")
col_desc = {
    'PRECTOTCORR': 'Daily Rainfall (mm)',
    'RH2M': 'Relative Humidity at 2m (%)',
    'T2MDEW': 'Dew Point Temperature (°C)',
    'T2M_MAX': 'Max Temperature (°C)',
    'T2M_MIN': 'Min Temperature (°C)',
    'T2MWET': 'Wet Bulb Temperature (°C)',
    'WS50M': 'Wind Speed at 50m (m/s)',
    'WD50M': 'Wind Direction at 50m (°)',
    'PS': 'Surface Pressure (kPa)',
    'QV2M': 'Specific Humidity at 2m (g/kg)',
    'ALLSKY_SFC_UV_INDEX': 'UV Index',
    'TS': 'Earth Skin Temperature (°C)',
}
for k, v in col_desc.items():
    print(f"  {k:25s}: {v}")


## 2. Statistical Summary

In [ ]:
key_cols = ['PRECTOTCORR','RH2M','T2M_MAX','T2M_MIN','T2MWET',
            'WS50M','QV2M','PS','ALLSKY_SFC_UV_INDEX','TS']
df[key_cols].describe().T.style.background_gradient(cmap='Blues').format(precision=2)


## 3. Rainfall Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Daily rainfall distribution (log scale)
rain_nonzero = df[df['PRECTOTCORR'] > 0]['PRECTOTCORR']
axes[0].hist(rain_nonzero, bins=60, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].axvline(rain_nonzero.mean(), color='red', linestyle='--', lw=2,
                label=f"Mean: {rain_nonzero.mean():.2f} mm")
axes[0].axvline(rain_nonzero.median(), color='yellow', linestyle='--', lw=2,
                label=f"Median: {rain_nonzero.median():.2f} mm")
axes[0].set_title('Daily Rainfall Distribution (Rain Days Only)', fontweight='bold')
axes[0].set_xlabel('Rainfall (mm)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Rain days vs dry days
rain_days = (df['PRECTOTCORR'] > 0).sum()
dry_days  = (df['PRECTOTCORR'] == 0).sum()
rain_label = 'Rain Days' + chr(10) + f'({rain_days:,})'
dry_label  = 'Dry Days'  + chr(10) + f'({dry_days:,})'
axes[1].pie([rain_days, dry_days],
            labels=[rain_label, dry_label],
            autopct='%1.1f%%',
            colors=['#3498db', '#e74c3c'],
            startangle=90, textprops={'fontsize':12})
axes[1].set_title('Rain Days vs Dry Days', fontweight='bold')

# Rainfall intensity categories
df['Rain_Category'] = pd.cut(df['PRECTOTCORR'],
    bins=[-0.01, 0, 2.5, 7.5, 35.5, 64.5, 115.5, 1000],
    labels=['No Rain','Light','Moderate','Heavy','Very Heavy','Extremely Heavy','Catastrophic'])
cat_counts = df['Rain_Category'].value_counts().reindex(
    ['No Rain','Light','Moderate','Heavy','Very Heavy','Extremely Heavy','Catastrophic'])
colors_cat = ['#95a5a6','#3498db','#2ecc71','#f39c12','#e74c3c','#8e44ad','#2c3e50']
bars = axes[2].bar(cat_counts.index, cat_counts.values, color=colors_cat, edgecolor='white')
axes[2].set_title('Rainfall Intensity Category Distribution', fontweight='bold')
axes[2].set_ylabel('Days Count')
axes[2].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, cat_counts.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Rainfall Distribution Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/rainfall_distribution.png', bbox_inches='tight', dpi=150)
plt.show()


## 4. Annual Rainfall Trends (2005–2025)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 7))

# Annual total rainfall across all districts
annual_total = df.groupby('YEAR')['PRECTOTCORR'].sum().reset_index()
axes[0].bar(annual_total['YEAR'], annual_total['PRECTOTCORR']/1000,
            color=sns.color_palette('Blues_r', len(annual_total)), edgecolor='white')
z = np.polyfit(annual_total['YEAR'], annual_total['PRECTOTCORR'], 1)
p = np.poly1d(z)
axes[0].plot(annual_total['YEAR'], p(annual_total['YEAR'])/1000,
             color='red', linewidth=2, linestyle='--', label='Trend')
axes[0].set_title('Annual Total Rainfall Across UP (2005–2025)', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Rainfall (1000 mm)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Annual avg temp trend
annual_temp = df.groupby('YEAR')[['T2M_MAX','T2M_MIN']].mean().reset_index()
axes[1].plot(annual_temp['YEAR'], annual_temp['T2M_MAX'],
             color='#e74c3c', linewidth=2.5, marker='o', markersize=6, label='Avg Max Temp')
axes[1].plot(annual_temp['YEAR'], annual_temp['T2M_MIN'],
             color='#3498db', linewidth=2.5, marker='s', markersize=6, label='Avg Min Temp')
axes[1].fill_between(annual_temp['YEAR'], annual_temp['T2M_MIN'],
                     annual_temp['T2M_MAX'], alpha=0.15, color='gray')
axes[1].set_title('Annual Temperature Trend (°C)', fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Temperature (°C)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.suptitle('Annual Trends (2005–2025)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/annual_trends.png', bbox_inches='tight', dpi=150)
plt.show()


## 5. Seasonal Rainfall & Climate Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

season_order = ['Winter','Pre-Monsoon','Monsoon','Post-Monsoon']
colors_s = ['#3498db','#f39c12','#2ecc71','#e74c3c']

# Avg rainfall by season
season_rain = df.groupby('SEASON')['PRECTOTCORR'].mean().reindex(season_order)
bars = axes[0].bar(season_order, season_rain.values, color=colors_s, edgecolor='white')
axes[0].set_title('Avg Daily Rainfall by Season (mm)', fontweight='bold')
axes[0].set_ylabel('Avg Rainfall (mm)')
for bar, val in zip(bars, season_rain.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val:.2f}mm', ha='center', va='bottom', fontsize=11)

# Monthly avg rainfall
monthly_rain = df.groupby('MO')['PRECTOTCORR'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
colors_mo = ['#3498db','#3498db','#f39c12','#f39c12','#f39c12',
             '#2ecc71','#2ecc71','#2ecc71','#2ecc71','#e74c3c','#e74c3c','#3498db']
bars2 = axes[1].bar(month_labels, monthly_rain.values, color=colors_mo, edgecolor='white')
axes[1].set_title('Avg Monthly Rainfall (mm)', fontweight='bold')
axes[1].set_ylabel('Avg Rainfall (mm)')
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars2, monthly_rain.values):
    if val > 0.5:
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                     f'{val:.1f}', ha='center', va='bottom', fontsize=9)

# Seasonal temperature
season_temp = df.groupby('SEASON')[['T2M_MAX','T2M_MIN']].mean().reindex(season_order)
x = range(len(season_order))
w = 0.35
axes[2].bar([i-w/2 for i in x], season_temp['T2M_MAX'], w,
            label='Max Temp', color='#e74c3c', alpha=0.85, edgecolor='white')
axes[2].bar([i+w/2 for i in x], season_temp['T2M_MIN'], w,
            label='Min Temp', color='#3498db', alpha=0.85, edgecolor='white')
axes[2].set_xticks(list(x))
axes[2].set_xticklabels(season_order)
axes[2].set_title('Avg Temperature by Season (°C)', fontweight='bold')
axes[2].set_ylabel('Temperature (°C)')
axes[2].legend()

plt.suptitle('Seasonal Climate Patterns', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/seasonal_analysis.png', bbox_inches='tight', dpi=150)
plt.show()


## 6. District-Wise Rainfall Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 12))

# Total rainfall by district (top 30)
district_rain = df.groupby('DISTRICT')['PRECTOTCORR'].sum().sort_values(ascending=False)
top30 = district_rain.head(30)
overall_avg = district_rain.mean()
colors_d = ['#2ecc71' if v > overall_avg else '#e74c3c' for v in top30.values]
axes[0].barh(top30.index[::-1], top30.values[::-1]/1000, color=colors_d[::-1])
axes[0].axvline(overall_avg/1000, color='gray', linestyle='--', lw=1.5,
                label=f'Avg: {overall_avg/1000:.1f}K mm')
axes[0].set_title('Total Rainfall by District (2005–2025)', fontweight='bold')
axes[0].set_xlabel('Total Rainfall (1000 mm)')
axes[0].legend()

# Avg daily rainfall by district (top 30)
district_avg = df.groupby('DISTRICT')['PRECTOTCORR'].mean().sort_values(ascending=False)
top30_avg = district_avg.head(30)
colors_da = sns.color_palette('Blues_r', len(top30_avg))
axes[1].barh(top30_avg.index[::-1], top30_avg.values[::-1], color=colors_da[::-1])
axes[1].set_title('Avg Daily Rainfall by District (mm)', fontweight='bold')
axes[1].set_xlabel('Avg Daily Rainfall (mm)')
for i, val in enumerate(top30_avg.values[::-1]):
    axes[1].text(val+0.01, i, f'{val:.2f}', va='center', fontsize=8)

plt.suptitle('District-Wise Rainfall Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/district_analysis.png', bbox_inches='tight', dpi=150)
plt.show()


## 7. Temperature Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Max temp distribution
axes[0].hist(df['T2M_MAX'], bins=50, color='#e74c3c', edgecolor='white', alpha=0.85)
axes[0].axvline(df['T2M_MAX'].mean(), color='black', linestyle='--', lw=2,
                label=f"Mean: {df['T2M_MAX'].mean():.1f}°C")
axes[0].set_title('Max Temperature Distribution (°C)', fontweight='bold')
axes[0].set_xlabel('Max Temperature (°C)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Monthly temp range
monthly_max = df.groupby('MO')['T2M_MAX'].mean()
monthly_min = df.groupby('MO')['T2M_MIN'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].plot(month_labels, monthly_max.values, color='#e74c3c',
             linewidth=2.5, marker='o', markersize=7, label='Avg Max Temp')
axes[1].plot(month_labels, monthly_min.values, color='#3498db',
             linewidth=2.5, marker='s', markersize=7, label='Avg Min Temp')
axes[1].fill_between(month_labels, monthly_min.values, monthly_max.values,
                     alpha=0.15, color='gray')
axes[1].set_title('Monthly Temperature Range (°C)', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Temperature (°C)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

# Temp vs Rainfall scatter
sample = df.sample(5000, random_state=42)
scatter = axes[2].scatter(sample['T2M_MAX'], sample['PRECTOTCORR'],
                          c=sample['RH2M'], cmap='Blues', alpha=0.4, s=15)
plt.colorbar(scatter, ax=axes[2], label='Humidity (%)')
axes[2].set_title('Max Temp vs Rainfall (colour = Humidity)', fontweight='bold')
axes[2].set_xlabel('Max Temperature (°C)')
axes[2].set_ylabel('Daily Rainfall (mm)')

plt.suptitle('Temperature Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/temperature_analysis.png', bbox_inches='tight', dpi=150)
plt.show()


## 8. Humidity, Wind & Atmospheric Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Humidity distribution
axes[0].hist(df['RH2M'], bins=50, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[0].axvline(df['RH2M'].mean(), color='red', linestyle='--', lw=2,
                label=f"Mean: {df['RH2M'].mean():.1f}%")
axes[0].set_title('Relative Humidity Distribution (%)', fontweight='bold')
axes[0].set_xlabel('Relative Humidity (%)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Monthly humidity
monthly_hum = df.groupby('MO')['RH2M'].mean()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
axes[1].plot(month_labels, monthly_hum.values, color='#9b59b6',
             linewidth=2.5, marker='o', markersize=7)
axes[1].fill_between(month_labels, monthly_hum.values, alpha=0.2, color='#9b59b6')
axes[1].set_title('Avg Monthly Relative Humidity (%)', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Humidity (%)')
axes[1].tick_params(axis='x', rotation=30)

# Wind speed by season
season_wind = df.groupby('SEASON')['WS50M'].mean().reindex(
    ['Winter','Pre-Monsoon','Monsoon','Post-Monsoon'])
colors_w = ['#3498db','#f39c12','#2ecc71','#e74c3c']
bars = axes[2].bar(season_wind.index, season_wind.values, color=colors_w, edgecolor='white')
axes[2].set_title('Avg Wind Speed by Season (m/s)', fontweight='bold')
axes[2].set_ylabel('Wind Speed (m/s)')
for bar, val in zip(bars, season_wind.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=11)

plt.suptitle('Humidity, Wind & Atmospheric Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/humidity_wind.png', bbox_inches='tight', dpi=150)
plt.show()


## 9. Monsoon Deep Dive

In [ ]:
monsoon = df[df['SEASON'] == 'Monsoon']

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Annual monsoon rainfall
annual_monsoon = monsoon.groupby('YEAR')['PRECTOTCORR'].sum()
bars = axes[0].bar(annual_monsoon.index, annual_monsoon.values,
                   color=sns.color_palette('Blues_r', len(annual_monsoon)), edgecolor='white')
axes[0].axhline(annual_monsoon.mean(), color='red', linestyle='--', lw=2,
                label=f'Avg: {annual_monsoon.mean():,.0f} mm')
axes[0].set_title('Annual Monsoon Rainfall Across UP', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Total Monsoon Rainfall (mm)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Month-wise monsoon rainfall (Jun-Sep)
monthly_monsoon = df[df['MO'].isin([6,7,8,9])].groupby('MO')['PRECTOTCORR'].mean()
month_labels_m = ['June','July','August','September']
colors_mm = ['#74b9ff','#0984e3','#2980b9','#74b9ff']
bars2 = axes[1].bar(month_labels_m, monthly_monsoon.values, color=colors_mm, edgecolor='white')
axes[1].set_title('Avg Monsoon Month Rainfall (mm)', fontweight='bold')
axes[1].set_ylabel('Avg Daily Rainfall (mm)')
for bar, val in zip(bars2, monthly_monsoon.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val:.2f}mm', ha='center', va='bottom', fontsize=11)

# Monsoon rainfall by district (top 20)
monsoon_district = monsoon.groupby('DISTRICT')['PRECTOTCORR'].sum().sort_values(ascending=False).head(20)
axes[2].barh(monsoon_district.index[::-1], monsoon_district.values[::-1]/1000,
             color=sns.color_palette('Blues_r', 20)[::-1])
axes[2].set_title('Top 20 Districts — Monsoon Rainfall (1000 mm)', fontweight='bold')
axes[2].set_xlabel('Monsoon Rainfall (1000 mm)')

plt.suptitle('Monsoon Season Deep Dive', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/monsoon_analysis.png', bbox_inches='tight', dpi=150)
plt.show()


## 10. Correlation Heatmap

In [ ]:
num_cols = ['PRECTOTCORR','RH2M','T2MDEW','QV2M','WS50M',
            'T2M_MAX','T2M_MIN','T2MWET','ALLSKY_SFC_UV_INDEX','TS','PS']

corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5,
            annot_kws={'size':11}, square=True,
            cbar_kws={'shrink':0.8})
ax.set_title('Correlation Matrix — Climate Variables', fontsize=14,
             fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('outputs/correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()


## 11. Extreme Weather Events Analysis

In [ ]:
# Extreme rainfall days (>64.5mm = Very Heavy or above)
extreme = df[df['PRECTOTCORR'] >= 64.5].copy()

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Extreme rain events by year
extreme_yr = extreme.groupby('YEAR').size()
axes[0].bar(extreme_yr.index, extreme_yr.values,
            color=sns.color_palette('Reds_r', len(extreme_yr)), edgecolor='white')
axes[0].set_title('Extreme Rainfall Events by Year (>64.5mm)', fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Extreme Events')
axes[0].tick_params(axis='x', rotation=45)

# Extreme rain by month
extreme_mo = extreme.groupby('MO').size()
axes[1].bar(range(1,13), [extreme_mo.get(m,0) for m in range(1,13)],
            color='#e74c3c', edgecolor='white', alpha=0.85)
month_labels = ['J','F','M','A','M','J','J','A','S','O','N','D']
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(month_labels)
axes[1].set_title('Extreme Rainfall Events by Month', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Number of Extreme Events')

# Top 10 districts for extreme events
extreme_dist = extreme.groupby('DISTRICT').size().sort_values(ascending=False).head(10)
axes[2].barh(extreme_dist.index[::-1], extreme_dist.values[::-1], color='#e74c3c')
axes[2].set_title('Top 10 Districts — Extreme Rain Events', fontweight='bold')
axes[2].set_xlabel('Number of Extreme Events')
for i, val in enumerate(extreme_dist.values[::-1]):
    axes[2].text(val+0.1, i, str(val), va='center', fontsize=9)

plt.suptitle('Extreme Weather Events Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/extreme_events.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Total extreme rainfall events >64.5mm: {len(extreme):,}")


## 12. Key Insights Summary

In [ ]:
total_records    = len(df)
rain_days_pct    = (df['PRECTOTCORR'] > 0).mean() * 100
avg_daily_rain   = df['PRECTOTCORR'].mean()
max_single_rain  = df['PRECTOTCORR'].max()
max_rain_date    = df.loc[df['PRECTOTCORR'].idxmax(), ['YEAR','MO','DY','DISTRICT']]
rainiest_dist    = df.groupby('DISTRICT')['PRECTOTCORR'].sum().idxmax()
driest_dist      = df.groupby('DISTRICT')['PRECTOTCORR'].sum().idxmin()
avg_max_temp     = df['T2M_MAX'].mean()
avg_min_temp     = df['T2M_MIN'].mean()
monsoon_contrib  = df[df['SEASON']=='Monsoon']['PRECTOTCORR'].sum() / df['PRECTOTCORR'].sum() * 100
extreme_count    = (df['PRECTOTCORR'] >= 64.5).sum()

lines = [
    "+---------------------------------------------------------------+",
    "|       UP RAINFALL & WEATHER 2005-2025 -- KEY INSIGHTS        |",
    "+---------------------------------------------------------------+",
    f"|  Total Records        : {total_records:,}                      |",
    f"|  Districts Covered    : {df['DISTRICT'].nunique()} districts                      |",
    f"|  Rain Days            : {rain_days_pct:.1f}% of all days                    |",
    f"|  Avg Daily Rainfall   : {avg_daily_rain:.2f} mm                          |",
    f"|  Max Single Day Rain  : {max_single_rain:.1f} mm                        |",
    f"|  Rainiest District    : {rainiest_dist:<30}   |",
    f"|  Driest District      : {driest_dist:<30}   |",
    f"|  Avg Max Temperature  : {avg_max_temp:.1f}°C                           |",
    f"|  Avg Min Temperature  : {avg_min_temp:.1f}°C                           |",
    f"|  Monsoon Contribution : {monsoon_contrib:.1f}% of annual rainfall          |",
    f"|  Extreme Events (>64mm): {extreme_count:,} events                     |",
    "+---------------------------------------------------------------+",
]
print(chr(10).join(lines))


---
## EDA Complete!

**Next Steps:**
- Run the Streamlit dashboard: `streamlit run app.py`
- Star this repo on GitHub!

> Built with Python, Pandas, Matplotlib & Seaborn
